# 05 - 决策树与随机森林

## 学习目标
- 理解决策树的分裂原理
- 掌握随机森林的集成思想
- 学会可视化决策树和特征重要性

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report

## 1. 决策树分类

In [ ]:
# 加载数据
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

# 训练决策树
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print(f'准确率: {accuracy_score(y_test, y_pred):.4f}')
print('\n分类报告:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# 可视化决策树
plt.figure(figsize=(16, 10))
plot_tree(dt, feature_names=iris.feature_names,
          class_names=iris.target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('决策树结构 (max_depth=3)')
plt.tight_layout()
plt.show()

## 2. 决策树深度对性能的影响

In [ ]:
depths = range(1, 11)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, 'o-', label='训练集')
plt.plot(depths, test_scores, 's-', label='测试集')
plt.xlabel('最大深度')
plt.ylabel('准确率')
plt.title('决策树深度对性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 随机森林

In [ ]:
# 随机森林
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'随机森林准确率: {accuracy_score(y_test, y_pred):.4f}')

# 交叉验证
cv_scores = cross_val_score(rf, iris.data, iris.target, cv=5)
print(f'交叉验证: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# 特征重要性
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), np.array(iris.feature_names)[indices], rotation=45)
plt.xlabel('特征')
plt.ylabel('重要性')
plt.title('随机森林特征重要性')
plt.tight_layout()
plt.show()

## 4. 决策树 vs 随机森林

In [ ]:
# 乳腺癌数据集比较
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    cv = cross_val_score(model, data.data, data.target, cv=5).mean()
    results.append({'Model': name, 'Train': train_acc, 'Test': test_acc, 'CV': cv})

print(pd.DataFrame(results).round(4))

## 小结
- 决策树直观但容易过拟合
- 随机森林通过集成降低方差
- 特征重要性帮助理解模型

---
下一个: [06_svm.ipynb](06_svm.ipynb)